# Ch10 — 無母數檢定 (Non-parametric Tests)

> **預計時長：** 1.5 小時  
> **資料集：** `datasets/titanic_train.csv`

## 學習目標

1. 理解無母數檢定 (non-parametric tests) 的適用時機與原理
2. 掌握 **Mann-Whitney U 檢定**（替代獨立樣本 t 檢定）
3. 掌握 **Wilcoxon Signed-Rank 檢定**（替代成對樣本 t 檢定）
4. 掌握 **Kruskal-Wallis H 檢定**（替代單因子 ANOVA）
5. 學會根據資料特性選擇適當的檢定方法

## 前置知識

本章建立在以下章節的基礎之上：

- **Ch01 假設檢定 (Hypothesis Testing)**：虛無假設、p-value、顯著水準
- **Ch03 ANOVA**：多組比較與事後檢定

如果尚未完成上述章節，建議先行閱讀。

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

print('環境準備完成')

In [ ]:
# 載入 Titanic 資料集
df = pd.read_csv('datasets/titanic_train.csv')

print(f'資料維度: {df.shape}')
print(f'欄位: {list(df.columns)}')
print()
df.head()

---

# Part A — 何時使用無母數檢定？

## 參數檢定的假設

我們在前幾章學過的 t 檢定與 ANOVA 都屬於**參數檢定 (parametric tests)**，它們假設：

1. 資料來自**常態分佈 (normal distribution)**
2. 各組具有**同質變異數 (homogeneity of variance)**
3. 觀測值之間彼此**獨立**

當這些假設不成立時，我們需要**無母數檢定 (non-parametric tests)**：

| 適用情境 | 說明 |
| :--- | :--- |
| 常態性不成立 | 資料呈現嚴重偏態 (skewness) 或有離群值 |
| 順序資料 (ordinal data) | 例如滿意度評分 1-5，只有排序意義 |
| 小樣本 + 偏態 | n < 30 且分佈明顯不對稱 |
| 中位數比較 | 研究問題關心的是中位數而非平均數 |

## 參數 vs. 無母數檢定對照表

| 研究問題 | 參數檢定 (Parametric) | 無母數替代 (Non-parametric) |
| :--- | :--- | :--- |
| 兩獨立樣本比較 | Independent t-test | **Mann-Whitney U** |
| 成對樣本比較 | Paired t-test | **Wilcoxon Signed-Rank** |
| 多組比較 (>2) | One-way ANOVA | **Kruskal-Wallis H** |
| 相關性 | Pearson r | **Spearman rho** |

> **核心原理：** 無母數檢定不直接比較原始數值，而是將資料轉換為**秩次 (ranks)**，再基於秩次進行比較。這使得檢定不受分佈形狀的影響。

In [ ]:
# 用 Titanic 的 Fare 欄位示範「非常態」分佈
fare = df['Fare'].dropna()

# Shapiro-Wilk 常態性檢定
stat, p_value = stats.shapiro(fare.sample(n=min(len(fare), 500), random_state=42))
print(f'Shapiro-Wilk 檢定: W = {stat:.4f}, p-value = {p_value:.2e}')
print(f'結論: {"拒絕常態假設" if p_value < 0.05 else "無法拒絕常態假設"}')
print(f'\n偏態係數 (skewness): {fare.skew():.2f}')
print(f'峰態係數 (kurtosis): {fare.kurtosis():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 直方圖
axes[0].hist(fare, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(fare.mean(), color='red', linestyle='--', label=f'Mean = {fare.mean():.1f}')
axes[0].axvline(fare.median(), color='orange', linestyle='--', label=f'Median = {fare.median():.1f}')
axes[0].set_xlabel('Fare')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Fare Distribution (Right-skewed)')
axes[0].legend()

# QQ Plot
stats.probplot(fare, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Fare')

plt.tight_layout()
plt.show()

print('\nFare 分佈呈現明顯右偏，不滿足常態假設 → 適合使用無母數檢定')

---

# Part B — Mann-Whitney U 檢定

## 原理

**Mann-Whitney U 檢定** (也稱為 Wilcoxon rank-sum test) 是獨立樣本 t 檢定的無母數替代方案。

- **虛無假設 (H0)：** 兩組的分佈相同（秩次沒有差異）
- **對立假設 (H1)：** 兩組的分佈不同

**步驟：**
1. 將兩組資料合併後排序，賦予秩次 (rank)
2. 分別計算各組的秩次總和
3. 計算 U 統計量，判斷秩次分佈是否有系統性差異

> **優點：** 不需要常態假設，對離群值穩健 (robust)。

In [ ]:
# Mann-Whitney U: 比較男性 vs 女性的 Fare
male_fare = df.loc[df['Sex'] == 'male', 'Fare'].dropna()
female_fare = df.loc[df['Sex'] == 'female', 'Fare'].dropna()

print(f'男性 Fare: n={len(male_fare)}, median={male_fare.median():.2f}, mean={male_fare.mean():.2f}')
print(f'女性 Fare: n={len(female_fare)}, median={female_fare.median():.2f}, mean={female_fare.mean():.2f}')

# 執行 Mann-Whitney U 檢定
u_stat, p_value = stats.mannwhitneyu(male_fare, female_fare, alternative='two-sided')

print(f'\nMann-Whitney U 檢定結果:')
print(f'  U statistic = {u_stat:.1f}')
print(f'  p-value = {p_value:.4e}')
print(f'  結論: {"拒絕 H0 — 男女 Fare 有顯著差異" if p_value < 0.05 else "無法拒絕 H0"}')

In [ ]:
# 計算效果量 (Effect Size): Rank-biserial correlation
# r = 1 - (2U) / (n1 * n2)
n1, n2 = len(male_fare), len(female_fare)
r_rb = 1 - (2 * u_stat) / (n1 * n2)

print(f'Rank-biserial correlation (r): {r_rb:.4f}')
print()

# 效果量判斷標準
abs_r = abs(r_rb)
if abs_r < 0.1:
    size_label = '可忽略 (negligible)'
elif abs_r < 0.3:
    size_label = '小效果 (small)'
elif abs_r < 0.5:
    size_label = '中效果 (medium)'
else:
    size_label = '大效果 (large)'

print(f'效果量大小: {size_label}')
print(f'解讀: |r| < 0.1 可忽略, 0.1-0.3 小, 0.3-0.5 中, > 0.5 大')

### 解讀

Mann-Whitney U 檢定的 p-value 遠小於 0.05，表示男性與女性的票價分佈存在**統計顯著差異**。

結合效果量 (rank-biserial correlation)，我們可以量化這個差異的**實際意義**：

- 統計顯著 ≠ 實際重要
- 大樣本下，微小差異也可能統計顯著
- **永遠要報告效果量 (effect size)！**

In [ ]:
# 視覺化: 男女 Fare 比較
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
data_to_plot = [male_fare, female_fare]
bp = axes[0].boxplot(data_to_plot, labels=['Male', 'Female'], patch_artist=True,
                     boxprops=dict(alpha=0.7),
                     medianprops=dict(color='red', linewidth=2))
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('coral')
axes[0].set_ylabel('Fare')
axes[0].set_title(f'Fare by Sex\n(Mann-Whitney U p = {p_value:.2e})')

# Histogram overlay
axes[1].hist(male_fare, bins=40, alpha=0.5, label=f'Male (med={male_fare.median():.1f})',
             color='steelblue', density=True)
axes[1].hist(female_fare, bins=40, alpha=0.5, label=f'Female (med={female_fare.median():.1f})',
             color='coral', density=True)
axes[1].set_xlabel('Fare')
axes[1].set_ylabel('Density')
axes[1].set_title('Fare Distribution by Sex')
axes[1].legend()

plt.tight_layout()
plt.show()

---

# Part C — Wilcoxon Signed-Rank 檢定

## 原理

**Wilcoxon Signed-Rank 檢定** 是成對樣本 t 檢定 (paired t-test) 的無母數替代方案。

- **適用情境：** 同一組受試者的前後測量（如：介入前 vs 介入後）
- **虛無假設 (H0)：** 差異的中位數為 0
- **對立假設 (H1)：** 差異的中位數不為 0

**步驟：**
1. 計算每對觀測的差異值
2. 取差異的絕對值並排序，賦予秩次
3. 將正差異和負差異的秩次分別加總
4. 比較兩個秩次總和

In [ ]:
# 模擬: 教育介入前後的測驗成績
# (Titanic 資料沒有成對資料，因此模擬一個教育情境)
np.random.seed(42)
n_students = 30

# 介入前成績 (非常態分佈 — 使用 beta 分佈模擬偏態資料)
before = np.round(stats.beta.rvs(a=2, b=5, size=n_students) * 100)

# 介入後成績 (平均提升 5 分，但個體差異大)
improvement = stats.norm.rvs(loc=5, scale=8, size=n_students)
after = np.clip(np.round(before + improvement), 0, 100)

score_df = pd.DataFrame({
    'before': before,
    'after': after,
    'diff': after - before
})

print(f'介入前 — median: {np.median(before):.0f}, mean: {np.mean(before):.1f}')
print(f'介入後 — median: {np.median(after):.0f}, mean: {np.mean(after):.1f}')
print(f'差異   — median: {np.median(after - before):.1f}')
print()
score_df.describe().round(1)

In [ ]:
# Wilcoxon Signed-Rank 檢定
w_stat, p_value_w = stats.wilcoxon(before, after, alternative='two-sided')

print(f'Wilcoxon Signed-Rank 檢定結果:')
print(f'  W statistic = {w_stat:.1f}')
print(f'  p-value = {p_value_w:.4f}')
print(f'  結論: {"拒絕 H0 — 介入前後有顯著差異" if p_value_w < 0.05 else "無法拒絕 H0 — 介入前後無顯著差異"}')

# 效果量: r = Z / sqrt(N)
z_score = stats.norm.ppf(p_value_w / 2)
r_effect = abs(z_score) / np.sqrt(n_students)
print(f'\n效果量 r = {r_effect:.3f}')

# 視覺化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 前後比較
for i in range(n_students):
    color = 'green' if after[i] > before[i] else 'red'
    axes[0].plot([0, 1], [before[i], after[i]], color=color, alpha=0.4)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Before', 'After'])
axes[0].set_ylabel('Score')
axes[0].set_title(f'Individual Score Changes\n(Wilcoxon p = {p_value_w:.4f})')

# 差異分佈
diff = after - before
axes[1].hist(diff, bins=15, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='No change')
axes[1].axvline(np.median(diff), color='orange', linestyle='--', linewidth=2,
                label=f'Median diff = {np.median(diff):.1f}')
axes[1].set_xlabel('Score Difference (After - Before)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Score Differences')
axes[1].legend()

plt.tight_layout()
plt.show()

### 解讀

Wilcoxon Signed-Rank 檢定直接比較成對觀測的差異秩次：

- 如果 p-value < 0.05，表示介入前後的成績存在**統計顯著差異**
- 效果量 r 幫助判斷差異的**實際大小** (0.1 小, 0.3 中, 0.5 大)
- 圖中綠色線代表進步、紅色線代表退步

> **注意：** Wilcoxon Signed-Rank 檢定要求差異值的分佈大致對稱。若差異值分佈嚴重偏態，可考慮使用 Sign test（符號檢定）。

---

# Part D — Kruskal-Wallis H 檢定

## 原理

**Kruskal-Wallis H 檢定** 是單因子 ANOVA 的無母數替代方案，用於比較**三組以上**的分佈。

- **虛無假設 (H0)：** 所有組的分佈相同
- **對立假設 (H1)：** 至少有一組的分佈與其他組不同

**步驟：**
1. 將所有組的資料合併後排序，賦予秩次
2. 計算各組的平均秩次
3. 計算 H 統計量（類似 ANOVA 的 F 統計量）
4. H 統計量近似卡方分佈 (chi-square distribution)

In [ ]:
# Kruskal-Wallis H: 比較三個船艙等級 (Pclass) 的 Fare
pclass_1 = df.loc[df['Pclass'] == 1, 'Fare'].dropna()
pclass_2 = df.loc[df['Pclass'] == 2, 'Fare'].dropna()
pclass_3 = df.loc[df['Pclass'] == 3, 'Fare'].dropna()

print('各等級 Fare 描述統計:')
for name, group in [('1st', pclass_1), ('2nd', pclass_2), ('3rd', pclass_3)]:
    print(f'  {name} class: n={len(group):3d}, median={group.median():7.2f}, mean={group.mean():7.2f}')

# 執行 Kruskal-Wallis H 檢定
h_stat, p_value_kw = stats.kruskal(pclass_1, pclass_2, pclass_3)

print(f'\nKruskal-Wallis H 檢定結果:')
print(f'  H statistic = {h_stat:.2f}')
print(f'  p-value = {p_value_kw:.4e}')
print(f'  結論: {"拒絕 H0 — 至少一組 Fare 有顯著差異" if p_value_kw < 0.05 else "無法拒絕 H0"}')

# 效果量: eta-squared (H) = (H - k + 1) / (N - k)
k = 3  # 組數
N = len(pclass_1) + len(pclass_2) + len(pclass_3)
eta_sq = (h_stat - k + 1) / (N - k)
print(f'\n效果量 eta-squared = {eta_sq:.4f} (0.01 小, 0.06 中, 0.14 大)')

### 解讀

Kruskal-Wallis H 檢定的結果告訴我們：**至少有一組**的 Fare 分佈與其他組不同。但它**不告訴我們是哪一組**。

為了找出具體是哪些組之間有差異，我們需要進行**事後檢定 (post-hoc test)**。

In [ ]:
# 事後檢定: 成對 Mann-Whitney U + Bonferroni 校正
pairs = [
    ('1st vs 2nd', pclass_1, pclass_2),
    ('1st vs 3rd', pclass_1, pclass_3),
    ('2nd vs 3rd', pclass_2, pclass_3),
]

n_comparisons = len(pairs)
alpha = 0.05
bonferroni_alpha = alpha / n_comparisons

print(f'Bonferroni 校正後的顯著水準: {bonferroni_alpha:.4f}')
print(f'比較次數: {n_comparisons}')
print(f'{"-" * 65}')

results = []
for name, g1, g2 in pairs:
    u, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    # Bonferroni 校正: p * n_comparisons
    p_adj = min(p * n_comparisons, 1.0)
    sig = 'Yes' if p_adj < alpha else 'No'
    # Rank-biserial
    r = 1 - (2 * u) / (len(g1) * len(g2))
    results.append({'Comparison': name, 'U': u, 'p_raw': p, 'p_adjusted': p_adj,
                    'Significant': sig, 'r_rb': r})
    print(f'{name:12s} | U={u:10.0f} | p_raw={p:.2e} | p_adj={p_adj:.2e} | sig={sig} | r={r:.3f}')

results_df = pd.DataFrame(results)
print(f'\n所有三組之間的 Fare 差異在 Bonferroni 校正後均顯著。')

### 事後檢定解讀

**Bonferroni 校正 (Bonferroni correction)** 是最保守的多重比較校正方法：

- 原理：將顯著水準除以比較次數 (alpha / k)
- 優點：簡單、控制 Type I error
- 缺點：過於保守，可能增加 Type II error

在本例中，三個船艙等級的票價存在顯著差異，這符合直覺 — 不同等級的票價本來就不同。

In [ ]:
# 視覺化: 三個等級的 Fare 分佈
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
bp = axes[0].boxplot([pclass_1, pclass_2, pclass_3],
                     labels=['1st Class', '2nd Class', '3rd Class'],
                     patch_artist=True,
                     medianprops=dict(color='red', linewidth=2))
colors = ['gold', 'lightblue', 'lightgreen']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_ylabel('Fare')
axes[0].set_title(f'Fare by Passenger Class\n(Kruskal-Wallis H = {h_stat:.1f}, p = {p_value_kw:.2e})')

# Violin plot (shows distribution shape)
parts = axes[1].violinplot([pclass_1, pclass_2, pclass_3],
                           positions=[1, 2, 3], showmeans=True, showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(colors[i])
    pc.set_alpha(0.7)
axes[1].set_xticks([1, 2, 3])
axes[1].set_xticklabels(['1st Class', '2nd Class', '3rd Class'])
axes[1].set_ylabel('Fare')
axes[1].set_title('Fare Distribution (Violin Plot)')

plt.tight_layout()
plt.show()

---

# Part E — 檢定選擇邏輯

## 決策流程圖

```
你要比較幾組？
|
|-- 2 組
|   |
|   |-- 獨立樣本？
|   |   |-- 是 → 常態? → 是 → Independent t-test
|   |   |              → 否 → Mann-Whitney U
|   |   |-- 否 (成對) → 常態? → 是 → Paired t-test
|   |                         → 否 → Wilcoxon Signed-Rank
|
|-- 3+ 組
    |
    |-- 獨立樣本？
        |-- 是 → 常態 + 同質變異數? → 是 → One-way ANOVA
        |                           → 否 → Kruskal-Wallis H
        |-- 否 (重複測量) → Friedman test
```

> **實務建議：** 當 n > 30 且分佈不太偏態時，參數檢定通常夠穩健 (robust)。但若有疑慮，使用無母數檢定更安全。

## 知識補給站 — Permutation Test (排列檢定)

除了傳統的無母數檢定，現代統計學越來越常使用**排列檢定 (permutation test)**：

1. 計算觀測到的檢定統計量（如兩組的中位數差異）
2. 隨機打亂 (shuffle) 組別標籤，重新計算統計量
3. 重複步驟 2 數千次，建立**虛無分佈 (null distribution)**
4. 計算觀測統計量在虛無分佈中的位置 → p-value

**優點：**
- 不需要任何分佈假設
- 概念直觀：「如果組別標籤不重要，打亂後結果應該差不多」
- 適用於任何檢定統計量

**缺點：**
- 計算量大（但以現代電腦來說不是問題）
- 需要可交換性 (exchangeability) 假設

```python
# 範例: Permutation test 虛擬碼
observed_diff = median(group_A) - median(group_B)
null_diffs = []
for _ in range(10000):
    shuffled = np.random.permutation(all_data)
    null_diffs.append(median(shuffled[:n_A]) - median(shuffled[n_A:]))
p_value = np.mean(np.abs(null_diffs) >= np.abs(observed_diff))
```

## 無母數檢定的代價

無母數檢定雖然假設較少，但並非萬能：

| 面向 | 說明 |
| :--- | :--- |
| **檢定力 (Power)** | 當資料確實是常態分佈時，無母數檢定的檢定力比參數檢定低約 5-15% |
| **效率損失** | 將連續資料轉為秩次會丟失部分資訊 |
| **信賴區間** | 無母數方法的信賴區間通常較寬 |
| **多因子分析** | 無母數方法在處理交互作用 (interaction) 時較不方便 |

> **結論：** 如果資料滿足常態假設，優先使用參數檢定。如果不確定，可以兩種方法都做，若結論一致則更有信心。

---

# 重點整理

| 檢定方法 | 用途 | scipy 函式 | 替代的參數檢定 |
| :--- | :--- | :--- | :--- |
| **Mann-Whitney U** | 兩獨立組比較 | `stats.mannwhitneyu()` | Independent t-test |
| **Wilcoxon Signed-Rank** | 成對樣本比較 | `stats.wilcoxon()` | Paired t-test |
| **Kruskal-Wallis H** | 多組比較 (>2) | `stats.kruskal()` | One-way ANOVA |

### 核心觀念

1. **無母數檢定**基於**秩次 (ranks)**，不假設特定分佈
2. 適用於**非常態資料、順序資料、小樣本偏態資料**
3. 永遠要報告**效果量 (effect size)**，而不僅僅依賴 p-value
4. **Bonferroni 校正**可控制多重比較的 Type I error
5. 當資料滿足常態假設時，參數檢定的**檢定力更高**

---

# 練習題

1. **Mann-Whitney U 練習：** 比較 Titanic 中存活者 (Survived=1) 與未存活者 (Survived=0) 的 Age 是否有顯著差異。
   - 提示：先用 `stats.shapiro()` 檢查常態性
   - 計算 rank-biserial effect size

2. **Kruskal-Wallis 練習：** 比較不同登船港口 (Embarked: S, C, Q) 的 Fare 是否有顯著差異。
   - 若顯著，進行事後 Mann-Whitney U + Bonferroni 校正
   - 繪製 boxplot 視覺化

3. **方法選擇練習：** 對於以下每個情境，說明你會使用哪種檢定方法，並解釋原因：
   - (a) 比較兩家餐廳的顧客滿意度（1-5 分量表）
   - (b) 比較同一批學生期中考與期末考的成績差異
   - (c) 比較四個部門的員工薪資中位數
   - (d) 比較藥物組 vs 安慰劑組的血壓數值（確認為常態分佈）

---

## TODO

- [ ] 延伸學習：Friedman test（重複測量的無母數替代）
- [ ] 延伸學習：Bootstrap 方法與排列檢定的實作
- [ ] 實作練習題 1-3
- [ ] 探索 `scipy.stats` 中其他無母數檢定函式

---

## 導覽列

| 上一章 | 目前 | 下一章 |
| :--- | :---: | ---: |
| [04 A/B Testing](04_AB_testing_procedure.ipynb) | **Ch10 無母數檢定** | (Coming Soon) |

---
*本教材為 iSpan Python 資料分析系列課程的一部分。*